# Fase 3

# Proyecto de Magíster: Detección de Anomalías y Posibles Fraudes en Permisos de Circulación

## Problemática
En la administración municipal, el cobro de los Permisos de Circulación Vehicular depende fuertemente de datos ingresados manualmente y tasaciones. Debido a la falta de validación estandarizada en los sistemas locales, existen múltiples errores de digitación, inconsistencias de formato y, potencialmente, **fraudes y evasiones** (por ejemplo, vehículos de alto valor comercial registrados con características alteradas para pagar menos, o modificaciones vehiculares no declaradas). Esta desestructuración de la información genera ineficiencias en la gestión pública y grandes pérdidas de ingresos municipales.

## Objetivo General
Diseñar e implementar soluciones algorítmicas en Python aplicadas al proyecto transversal, integrando principios de programación estructurada, recursiva y orientada a objetos para construir código modular, reutilizable y escalable.

In [7]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import unicodedata
import re

from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler

### Definimos todas las funciones de exploracion en las clase Explorador

In [ ]:
class Explorador:
    """Clase para realizar un análisis exploratorio inicial del dataset, 
    enfocándose en la estructura, calidad y posibles anomalías.
    Cada método privado se encarga de un aspecto específico del análisis, 
    y el método público 'explorar' los ejecuta en secuencia."""

    def __init__(self, df):
        self.df = df

    def _explorar_estructura(self):
        """Muestra la estructura básica del DataFrame, incluyendo dimensiones,
        tipos de datos y estadísticas descriptivas."""
        print(f"Dimensiones: {self.df.shape[0]} filas x {self.df.shape[1]} columnas\n")
        display(self.df.head(10))
        print("\n--- Tipos de datos ---")
        display(self.df.dtypes.rename('tipo').to_frame())
        print("\n--- Estadísticas descriptivas ---")
        display(self.df.describe())

    def _analizar_categorias(self, columnas=None):
        """Muestra las frecuencias de las columnas categóricas."""
        if columnas is None:
            columnas = ['Metodo de Pago', 'Cuotas Permiso', 'TipoVehiculo', 'Combustible', 'Transmision']
        print("--- Frecuencias por columna categórica ---")
        for col in columnas:
            print(f"\n{col}:")
            print(self.df[col].value_counts(dropna=False))
            print("-" * 30)

    def _generar_reporte_nulos(self):
        """Genera un reporte detallado de las columnas con valores nulos, 
        incluyendo el conteo y porcentaje de nulos."""
        reporte = []
        for col in self.df.columns:
            nulos = self.df[col].isnull().sum()
            reporte.append({
                'Columna': col,
                'Tipo': str(self.df[col].dtype),
                'Nulos': nulos,
                '% Nulos': f"{nulos / len(self.df) * 100:.2f}%"
            })
        df_reporte = pd.DataFrame(reporte)
        con_nulos = df_reporte[df_reporte['Nulos'] > 0].sort_values('Nulos', ascending=False)
        print("--- Columnas con valores nulos ---")
        display(con_nulos)
        return df_reporte

    def _detectar_duplicados(self):
        """Detecta filas duplicadas en el DataFrame."""
        n = self.df.duplicated().sum()
        print(f"Filas totalmente duplicadas: {n}")
        if n > 0:
            display(self.df[self.df.duplicated(keep=False)].sort_values('_id').head(10))
        return n

    def _detectar_tipos_mixtos(self, columnas_numericas):
        """ Detecta valores no numéricos en columnas que deberían ser numéricas """
        print("--- Detección de tipos mixtos en columnas numéricas ---")
        for col in columnas_numericas:
            temp = pd.to_numeric(self.df[col], errors='coerce')
            mascara = self.df[col].notnull() & temp.isnull()
            if mascara.any():
                print(f"  [!] '{col}': {self.df[mascara][col].unique()} ({mascara.sum()} filas)")
            else:
                print(f"  [OK] '{col}'")

    def _analizar_anomalias_texto(self):
        """ Analiza la longitud de las cadenas en columnas de 
        texto para detectar posibles anomalías o inconsistencias. """
        print("--- Longitud máxima en columnas de texto ---")
        for col in ['TipoVehiculo', 'Marca', 'Modelo']:
            max_len = self.df[col].astype(str).map(len).max()
            print(f"  '{col}': {max_len} caracteres")
            if max_len > 50:
                display(self.df[self.df[col].astype(str).map(len) > 50][[col]].head())
        print("\n--- Cilindrada promedio por TipoVehiculo ---")
        resumen = self.df.groupby('TipoVehiculo')['Cilindrada'].agg(['mean', 'max', 'min'])
        display(resumen.sort_values('mean', ascending=False))

    def _validar_coherencia_fechas(self):
        """Valida que 'Fecha_Emision' no sea posterior a 'Fecha_Vencimiento'."""
        fe = pd.to_datetime(self.df['Fecha_Emision'], errors='coerce')
        fv = pd.to_datetime(self.df['Fecha_Vencimiento'], errors='coerce')
        mascara = fe > fv
        n = mascara.sum()
        print(f"Registros con Fecha_Emision > Fecha_Vencimiento: {n}")
        if n > 0:
            cols = ['_id', 'Fecha_Emision', 'Fecha_Vencimiento', 'Ano_Fabricacion', 'Marca', 'Modelo']
            display(self.df[mascara][cols].head(10))
            plt.figure(figsize=(10, 5))
            sns.histplot(self.df[mascara]['Ano_Fabricacion'], bins=15, kde=True, color='orange')
            plt.title('Año de Fabricación — Registros con Fechas Inconsistentes')
            plt.xlabel('Año de Fabricación')
            plt.ylabel('Frecuencia')
            plt.grid(axis='y', linestyle='--', alpha=0.7)
            plt.tight_layout()
            plt.show()
        return n


### Definimos todas las funciones de limpieza en las clase Limpieza

In [ ]:
class Limpieza:
    """Objeto que encapsula todos los pasos de limpieza del DataFrame.

    El DataFrame se pasa en el constructor y cada metodo privado aplica
    una transformacion especifica. El metodo publico limpiar() los
    ejecuta todos en orden.
    """

    def __init__(self, df):
        self.df = df.copy()


    @staticmethod
    def _eliminar_acentos(texto):
        """Elimina acentos y caracteres diacríticos de una cadena de texto """
        if not isinstance(texto, str):
            return texto
        return "".join(
            c for c in unicodedata.normalize('NFD', texto)
            if not unicodedata.combining(c)
        )

    def _eliminar_columna_equipamiento(self):
        """Elimina la columna 'Equipamiento' si existe, debido a su alta proporción de valores nulos o irrelevantes."""
        # >55% sin información útil (nulos + 'EQUI' sin definición)
        self.df = self.df.drop(columns=['Equipamiento'], errors='ignore')

    def _eliminar_columna_tonelaje(self):
        """Elimina la columna 'Tonelaje' si existe, debido a su alta proporción de registros con valor 0, que no aportan información analítica."""
        # >98% de los registros tienen valor 0, sin aporte analítico
        self.df = self.df.drop(columns=['Tonelaje'], errors='ignore')

    def _convertir_numericas(self):
        """Los valores no convertibles pasan a NaN para ser imputados después."""
        columnas = ['Valor_Contado', 'Total_a_Pagar', 'Ano_Fabricacion', 'Cilindrada']
        for col in [c for c in columnas if c in self.df.columns]:
            antes = self.df[col].isna().sum()
            self.df[col] = pd.to_numeric(self.df[col], errors='coerce')
            convertidos = self.df[col].isna().sum() - antes
            if convertidos > 0:
                print(f"  [!] '{col}': {convertidos} valor(es) no numérico(s) → NaN")

    def _imputar_nulos(self):
        """Imputa valores nulos utilizando estrategias específicas para cada columna"""
        self.df['Combustible'] = self.df['Combustible'].fillna(self.df['Combustible'].mode()[0])
        self.df['Transmision'] = self.df['Transmision'].fillna(self.df['Transmision'].mode()[0])
        self.df['Cilindrada'] = (
            self.df.groupby('TipoVehiculo')['Cilindrada']
            .transform(lambda x: x.fillna(x.median()))
        )
        self.df['Cilindrada'] = self.df['Cilindrada'].fillna(self.df['Cilindrada'].median())
        self.df['Cilindrada'] = self.df['Cilindrada'].astype(int)
        self.df['Codigo_SII'] = self.df['Codigo_SII'].fillna('SIN_CODIGO')
        self.df['Comuna_Anterior'] = self.df['Comuna_Anterior'].fillna('OTRA')

    def _normalizar_texto(self):
        """ Aplica limpieza y normalización a las columnas de texto, incluyendo eliminación de espacios, conversión a mayúsculas y eliminación de acentos. """
        columnas = [
            'TipoVehiculo', 'Marca', 'Modelo', 'Combustible', 'Transmision',
            'Metodo de Pago', 'Cuotas Permiso', 'Comuna_Propietario',
            'Comuna_Permiso', 'Comuna_Anterior'
        ]
        for col in columnas:
            if col in self.df.columns:
                # Sin astype(str): los NaN se preservan como NaN en lugar de
                # convertirse al string "nan", que engañaría a la validación final.
                self.df[col] = (
                    self.df[col].str.strip().str.upper()
                    .apply(self._eliminar_acentos)
                )

    def _normalizar_fechas(self):
        """ Convierte las columnas de fecha a formato datetime, manejando errores de conversión."""
        for col in ['Fecha_Emision', 'Fecha_Vencimiento']:
            self.df[col] = pd.to_datetime(self.df[col], errors='coerce').dt.date

    def _consolidar_tipo_vehiculo(self):
        """ Consolida categorías similares en 'TipoVehiculo' para reducir la cardinalidad y mejorar la consistencia de los datos. """
        mapeo = {
            'SEDAN': 'AUTOMOVIL',      'SEDAN 2': 'AUTOMOVIL',
            'SUV': 'STATION WAGON',    'SUV 2': 'STATION WAGON',
            'MOTO1': 'MOTO',           'MOTO2': 'MOTO',
            'VAN 2': 'FURGON',
            'MINIBUS PARTICULAR':         'MINIBUS',
            'MINIBUS TRANS  PASAJERO':    'MINIBUS',
            'MINIBUS TRANS PASAJERO':     'MINIBUS',
            'MINIBUS DE TURISMO':         'MINIBUS',
            'MINIBUS ESCOLAR':            'MINIBUS',
        }
        self.df['TipoVehiculo'] = self.df['TipoVehiculo'].replace(mapeo)

    def _consolidar_combustible_transmision(self):
        """ Consolida categorías similares en 'Combustible' y 'Transmision' para mejorar la consistencia de los datos. """
        self.df['Transmision'] = self.df['Transmision'].replace({'CVT': 'AUT'})
        # 'MEC' es un error de digitación registrado en la columna Combustible
        self.df['Combustible'] = self.df['Combustible'].replace({'MEC': 'DIES'})



## 1. Preprocesamiento
Lectura del archivo CSV y construcción del DataFrame de trabajo.

In [ ]:
class Preprocesador:
    """Objeto que guarda una tabla de datos y sabe prepararla para el analisis.

    La tabla vive en el atributo self.df. Cada metodo (cargar, limpiar, codificar,
    escalar, validar) es un paso del pipeline que trabaja sobre esa misma tabla.
    """

    def __init__(self, ruta):
        # __init__ es el constructor: deja listos los atributos del objeto.
        self.ruta = ruta        # donde esta el archivo de datos
        self.df = None          # aqui guardaremos la tabla (todavia vacia)
        self.scaler = None      # aqui guardaremos el escalador (para reusarlo luego)

    def cargar_datos(self):
        """ Carga el dataset desde disco y lo guarda en self.df. 
        Verifica que el archivo exista antes de intentar cargarlo. """
        if not os.path.exists(self.ruta):
            raise FileNotFoundError(
                f"\n[ERROR] No se encontró el archivo: '{self.ruta}'\n"
                f"Verifica que el CSV esté en la ruta correcta antes de continuar."
            )
        self.df = pd.read_csv(self.ruta, encoding='utf-8')
        print(f"Dataset cargado exitosamente. Dimensiones: {self.df.shape[0]} filas x {self.df.shape[1]} columnas.")
        return self.df
    
    
    def explorar(self, columnas_numericas=None):
        """Realiza un análisis exploratorio inicial para entender la estructura y calidad del dataset."""
        if columnas_numericas is None:
            columnas_numericas = ['Valor_Libro', 'Monto_Permiso', 'Cilindrada', 'Ano_Fabricacion']

        self._explorar_estructura()
        self._analizar_categorias()
        self._generar_reporte_nulos()
        self._detectar_duplicados()
        self._detectar_tipos_mixtos(columnas_numericas)
        self._analizar_anomalias_texto()
        self._validar_coherencia_fechas()
        
    def limpiar(self):
        """Realiza la limpieza y transformación de los datos."""
        self._eliminar_columna_equipamiento()
        self._eliminar_columna_tonelaje()
        self._convertir_numericas()
        self._imputar_nulos()
        self._normalizar_texto()
        self._normalizar_fechas()
        self._consolidar_tipo_vehiculo()
        self._consolidar_combustible_transmision()
        print(f"Limpieza completada. Dimensiones finales: {self.df.shape[0]} filas x {self.df.shape[1]} columnas.")
        return self.df
    
    def codificar(self):
        """Convierte los nombres de las columnas a snake_case para facilitar su uso en análisis posteriores."""
        def _a_snake_case(nombre):          # helper definido adentro
            s = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', nombre)
            s = re.sub('([a-z0-9])([A-Z])', r'\1_\2', s).lower()
            return s.replace(' ', '_').replace('.', '').replace('__', '_').strip('_')

        self.df.columns = [_a_snake_case(col) for col in self.df.columns]
        print(f"Codificación completada — columnas: {self.df.columns.tolist()}")
        return self.df
    
    def validar(self, anio_actual=2026):
        """Verifica que el dataset cumple condiciones mínimas de calidad antes de exportar."""
        assert self.df.isna().sum().sum() == 0,                              "Existen valores nulos"
        assert not self.df.duplicated().any(),                               "Existen filas duplicadas"
        assert self.df['ano_fabricacion'].between(1900, anio_actual).all(),  "Años de fabricación fuera de rango"
        assert (self.df['cilindrada'] >= 0).all(),                           "Cilindradas negativas detectadas"
        print("Todas las validaciones pasaron correctamente.")
        return self.df
    
    def exportar(self, ruta):
        """Guarda el dataset procesado en disco."""
        os.makedirs(os.path.dirname(ruta), exist_ok=True)
        self.df.to_csv(ruta, index=False, encoding='utf-8')
        kb = os.path.getsize(ruta) / 1024
        print(f"Dataset exportado → {ruta}  ({kb:.1f} KB | {self.df.shape[0]} filas x {self.df.shape[1]} columnas)")

        